
# Batch Predict + Metrics & Confusion Matrix

This notebook:
- Loads a **Lightning checkpoint** for `IrrMLPClassifier`  
- Applies it to a set of CSVs (glob) with AlphaEarth features  
- If labels are present, computes **AUROC**, **AUPRC**, **ROC**/**PR** curves, and a **confusion matrix** at a chosen threshold  
- Gives per-year and per-county summaries

> **Notes**
> - Update the config cell with your `CKPT_PATH` and `DATA_GLOB`.
> - The code supports **CPU**, **CUDA**, and **Apple MPS** (M1/M2).


In [ ]:

# Imports
from pathlib import Path
import glob
import re
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    roc_curve,
    classification_report,
)

from irr.models.mlp_classifier import IrrMLPClassifier, ModelConfig
from irr.constants import FEATURES, LABEL_COL


In [ ]:

# ==== User Config ====
CKPT_PATH = "outputs/logs/mlp_classifier_tb/version_x/checkpoints/best.ckpt"  # <-- set me
DATA_GLOB = "raw_data/*.csv"                                                  # <-- set me
OUT_CSV   = "notebooks/predictions/preds_all.csv"                             # where to save predictions
THRESHOLD = 0.50                                                               # decision threshold
BATCH_SIZE = 4096

# Device pick
def pick_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = pick_device()
DEVICE


In [ ]:

# Load CSVs (keep all columns; require FEATURES present)
paths = sorted(glob.glob(DATA_GLOB))
if not paths:
    raise FileNotFoundError(f"No CSV files matched: {DATA_GLOB}")

dfs = []
for p in paths:
    df = pd.read_csv(p)
    missing = [c for c in FEATURES if c not in df.columns]
    if missing:
        raise ValueError(f"{p} missing feature columns: {missing}")
    df["_source_path"] = p  # keep origin for summaries
    dfs.append(df)

full = pd.concat(dfs, ignore_index=True)
full = full.dropna(subset=FEATURES).reset_index(drop=True)
print(f"Loaded {len(paths)} files → combined shape: {full.shape}")
full.head(3)


In [ ]:

# Load model
model: IrrMLPClassifier = IrrMLPClassifier.load_from_checkpoint(CKPT_PATH, cfg=ModelConfig())
model.eval().to(DEVICE)

# DataLoader for speed
X = torch.tensor(full[FEATURES].values, dtype=torch.float32)
ds = TensorDataset(X)
pin = (DEVICE == "cuda")
dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=pin)

# Inference
logits_list = []
with torch.no_grad():
    for (xb,) in dl:
        xb = xb.to(DEVICE, non_blocking=pin)
        logits = model(xb)  # (B,)
        logits_list.append(logits.detach().cpu().numpy())

import numpy as np
logits = np.concatenate(logits_list, axis=0)
probs = 1.0 / (1.0 + np.exp(-logits))  # sigmoid

preds_df = full.copy()
preds_df["logit_irrigated"] = logits
preds_df["prob_irrigated"] = probs
preds_df["pred_irrigated"] = (probs > THRESHOLD).astype(int)

# Save predictions
out_path = Path(OUT_CSV)
out_path.parent.mkdir(parents=True, exist_ok=True)
preds_df.to_csv(out_path, index=False)
print(f"Wrote predictions to {out_path.resolve()}")
print(f"Rows: {len(preds_df):,} | Threshold: {THRESHOLD:.3f} | Device: {DEVICE}")
preds_df.head(3)


In [ ]:

# Metrics (if labels are present)
has_labels = LABEL_COL in preds_df.columns
if has_labels:
    y_true = preds_df[LABEL_COL].astype(int).values
    y_prob = preds_df["prob_irrigated"].values
    y_pred = (y_prob > THRESHOLD).astype(int)

    auroc = roc_auc_score(y_true, y_prob)
    auprc = average_precision_score(y_true, y_prob)
    print(f"AUROC: {auroc:.4f}  |  AUPRC: {auprc:.4f}")

    # Confusion matrix at current threshold
    cm = confusion_matrix(y_true, y_pred)  # [[TN, FP], [FN, TP]]
    print(f"Confusion matrix (thr = {THRESHOLD:.3f}):\n{cm}")

    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, digits=4))

else:
    print("No labels found in data; skipping metrics.")


In [ ]:

# Plots
if has_labels:
    # Confusion Matrix
    fig, ax = plt.subplots(figsize=(3,3), dpi=120)
    ax.imshow(cm, interpolation="nearest")
    ax.set_title(f"Confusion Matrix @ thr={THRESHOLD:.2f}")
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(["Pred 0","Pred 1"])
    ax.set_yticklabels(["True 0","True 1"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(int(cm[i, j])), ha="center", va="center")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    fig.tight_layout()
    plt.show()

    # ROC
    from sklearn.metrics import roc_curve, precision_recall_curve
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.figure(figsize=(4,4), dpi=120)
    plt.plot(fpr, tpr, label=f"AUROC={auroc:.3f}")
    plt.plot([0,1],[0,1], linestyle="--")
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title("ROC Curve")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Precision-Recall
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    plt.figure(figsize=(4,4), dpi=120)
    plt.plot(rec, prec, label=f"AUPRC={auprc:.3f}")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title("Precision-Recall Curve")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:

# Best-F1 threshold
if has_labels:
    from sklearn.metrics import precision_recall_curve
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1 = (2 * prec[:-1] * rec[:-1]) / np.maximum(prec[:-1] + rec[:-1], 1e-9)
    best_idx = int(np.argmax(f1))
    best_thr = float(thr[best_idx])
    print(f"Best F1 at threshold ~ {best_thr:.3f}  (P={prec[best_idx]:.2f}, R={rec[best_idx]:.2f})")


In [ ]:

# Per-year & per-county breakdowns
if "year" not in preds_df.columns:
    preds_df["year"] = preds_df["_source_path"].astype(str).apply(
        lambda s: int(re.search(r"20\d{2}", s).group()) if re.search(r"20\d{2}", s) else -1
    )

if has_labels:
    by_year = preds_df.groupby("year")[LABEL_COL].value_counts().unstack(fill_value=0).sort_index()
    print("\nLabel counts by year (head):")
    display(by_year.head(10))

county_col = None
for cand in ["county", "county_name", "county_fips"]:
    if cand in preds_df.columns:
        county_col = cand
        break

if county_col:
    if has_labels:
        by_county = preds_df.groupby(county_col)[LABEL_COL].value_counts().unstack(fill_value=0)
        print("\nLabel counts by county (head):")
        display(by_county.head(10))

        def cm_counts(g: pd.DataFrame):
            yt = g[LABEL_COL].astype(int).values
            yp = (g["prob_irrigated"].values > THRESHOLD).astype(int)
            tn, fp, fn, tp = confusion_matrix(yt, yp).ravel()
            return pd.Series(dict(TN=tn, FP=fp, FN=fn, TP=tp, support=len(g)))

        county_cm = preds_df.groupby(county_col).apply(cm_counts).sort_values("support", ascending=False)
        print("\nPer-county confusion counts (head):")
        display(county_cm.head(10))
    else:
        print("\nNo labels found, showing raw predicted counts per county:")
        pred_counts = preds_df.groupby(county_col)["pred_irrigated"].value_counts().unstack(fill_value=0)
        display(pred_counts.head(10))
else:
    print("\nNo county column found; skipping county breakdowns.")
